In [20]:
# =============================================================================
# CELL 1: CONFIGURATION
# =============================================================================
# RAGU V3 Production Output -- Individual formula first, then average.
#   RAGU = MS + GL + Recovery + LTV + APR  (higher score = lower loss)
#   - GL: (1-LM) * UL/0.027, where UL = 0.9 - (MS-125)*0.027 - (72-term)/240
#   - Recovery: UL * F * R * 100  (ADDED -- higher recovery = lower net loss = higher RAGU)
#   - LTV: LTV_COEF * (baseline - actual)
#   - APR: (baseline - actual)/0.01 * APR_MULT

# --- Granularity: 'q' = quarterly, 'm' = monthly ---
granularity = 'q'

# --- Date Range ---
START_DATE = '2020-01-01'
END_DATE = None

# --- Query Control ---
run_every_query = False

# --- Date Column per Granularity ---
DATE_COL_MAP = {'q': 'book_date', 'm': 'book_date'}

# --- LOBs ---
LOBS = ['AN', 'FRN', 'STG', 'FLD', 'ENT', 'KMX']

# --- Rollup Groups ---
ROLLUP_GROUPS = {
    'Franchise Independent': ['AN', 'FLD', 'FRN', 'STG'],
    'nonKMX': ['AN', 'FRN', 'STG', 'FLD', 'ENT'],
    'POS': ['AN', 'FRN', 'STG', 'FLD', 'ENT', 'KMX'],
}

# --- Per-LOB Baselines ---
BASELINES = {
    'AN':  {'ltv': 1.94, 'apr': 0.25},
    'FRN': {'ltv': 1.94, 'apr': 0.25},
    'STG': {'ltv': 1.94, 'apr': 0.25},
    'FLD': {'ltv': 1.45, 'apr': 0.235},
    'ENT': {'ltv': 1.45, 'apr': 0.235},
    'KMX': {'ltv': 1.59, 'apr': 0.25},
}

# --- V3 Formula Parameters ---
UL_TO_MS = 0.027
UL_INTERCEPT = 0.9
UL_MS_CENTER = 125
UL_TERM_CENTER = 72
UL_TERM_DENOM = 240
RECOVERY_TO_SCORE = 100

APR_MULT = 0.8223
FIND_RATE = 0.75
LTV_COEF = 1.1

# --- KMX Rescaling ---
KMX_BPS_PER_POINT = 0.65
KMX_MS_OFFSET = 50

# --- MMI Vintage Adjustment (post-scoring market overlay) ---
MMI_ENABLED = False          # Set True to apply MMI market adjustment
MMI_LAG_MONTHS = 24          # Average months to repossession
MMI_COEF = 38.3089           # Auto-derived from validation: beta_mmi / beta_ragu
MMI_CSV = '../output/mmi_2019_seasonality.csv'

# --- Excel Output ---
EXCEL_OUTPUT = '../output/ragu_v3_production.xlsx'

In [21]:
# =============================================================================
# CELL 2: IMPORTS AND DERIVED CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import os
import openpyxl
from tqdm.notebook import tqdm

tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

date_col = DATE_COL_MAP[granularity]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()

PERIOD_FREQ_MAP = {'q': 'Q', 'm': 'M'}
period_freq = PERIOD_FREQ_MAP[granularity]

start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)

min_date_sql = f"'{START_DATE}'"

print(f"Granularity: {granularity}")
print(f"Date column: {date_col}")
print(f"Period range: {start_period} to {end_period}")
print(f"SQL min_date: {min_date_sql}")

Granularity: q
Date column: book_date
Period range: 2020Q1 to 2026Q3
SQL min_date: '2020-01-01'


In [22]:
# =============================================================================
# CELL 3: RAGU V3 FORMULA DEFINITIONS
# =============================================================================

def compute_unit_loss(model_score, con_term):
    """UL = 0.9 - (MS - 125)*0.027 - (72 - term)/240"""
    return UL_INTERCEPT - (model_score - UL_MS_CENTER) * UL_TO_MS - (UL_TERM_CENTER - con_term) / UL_TERM_DENOM


def compute_gross_loss_impact(loss_multiplier, unit_loss):
    """GL impact: (1-LM) * UL / 0.027"""
    return (1 - loss_multiplier) * unit_loss / UL_TO_MS


def compute_recovery_impact(recovery_multiplier, unit_loss, find_rate=FIND_RATE):
    """Recovery impact: UL * F * R * 100"""
    return unit_loss * find_rate * recovery_multiplier * RECOVERY_TO_SCORE


def compute_ltv_impact(baseline_ltv, actual_ltv, ltv_coef=LTV_COEF):
    """LTV impact: LTV_COEF * (baseline - actual)"""
    return ltv_coef * (baseline_ltv - actual_ltv)


def compute_apr_impact(baseline_apr, actual_apr, apr_mult=APR_MULT):
    """APR impact: per 1% deviation from baseline."""
    return (baseline_apr - actual_apr) / 0.01 * apr_mult


def rescale_kmx(value, bps_per_point=KMX_BPS_PER_POINT, ms_offset=KMX_MS_OFFSET,
                is_model_score=False):
    """Rescale KMX from 65bps/pt to 100bps/pt space."""
    scaled = value * bps_per_point
    if is_model_score:
        scaled += ms_offset
    return scaled

In [23]:
# =============================================================================
# CELL 4: UTILITY FUNCTIONS
# =============================================================================

def run_sql(filename, sub_list=None, connection=None, filename_is_query=False,
            chunksize=200_000):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)

    def _read(conn):
        warnings.filterwarnings("ignore", category=UserWarning)
        chunks = pd.read_sql_query(sql=query, con=conn, chunksize=chunksize)
        df = pd.concat(chunks, ignore_index=True)
        warnings.filterwarnings("default", category=UserWarning)
        return df

    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            return _read(conn)
    else:
        return _read(connection)


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(filename, pickle_name, sub_list=None, connection=None,
               force_refresh=False, filename_is_query=False):
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(filename, sub_list=sub_list, connection=connection,
                     filename_is_query=filename_is_query)
        store_pickle(df, pickle_name)
        return df
    return get_pickle(pickle_name)


def weighted_average_and_sum(group, metrics):
    """Per-metric population-aware weighted average."""
    if isinstance(metrics, str):
        valid = group[metrics].notna()
        if valid.any():
            weighted_avg = (group.loc[valid, metrics] * group.loc[valid, 'amt_financed']).sum() / group.loc[valid, 'amt_financed'].sum()
        else:
            weighted_avg = np.nan
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        valid = group[metric].notna()
        if valid.any():
            result_dict[metric] = (
                (group.loc[valid, metric] * group.loc[valid, 'amt_financed']).sum()
                / group.loc[valid, 'amt_financed'].sum()
            )
        else:
            result_dict[metric] = np.nan
    return pd.Series(result_dict)


def assign_period(df, col_name, freq):
    """Assign a pd.Period column from a date column."""
    dt_series = pd.to_datetime(df[col_name])
    return dt_series.dt.to_period(freq)


def format_vintage(period_series):
    """Convert pd.Period series to formatted vintage strings."""
    if len(period_series) == 0:
        return period_series.astype(str)
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)

In [24]:
# =============================================================================
# CELL 5: ULA MULTIPLIER FUNCTIONS
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df, leave_out='None'):
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag
    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag
    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)
    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)
    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag
    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag
    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date
    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag
    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag
    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag
    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag
    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)
    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag
    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)
    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)

    return ula_df


def get_ula_multiplier_kmx(ula_df, loss_scale=0.067, leave_out='None'):
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Low FICO':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag))
    if leave_out != 'Low Vantage':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag))
    if leave_out != 'High PTI':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.05 * ula_df.high_pti_tier_1_flag
            + 0.1 * ula_df.high_pti_tier_2_flag
            + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag)
    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] /= (1 + loss_scale)

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + (-0.01 + 0.06 * ula_df.secured_credit_flag)
            * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag))
    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag
    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025
            + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108
            + ula_df.secured_credit_flag * 0.295)))
    if leave_out != 'Fraud':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + (ula_df.fraud_adjustment - 1)
    if leave_out != 'Clip':
        mask = (ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag)
        ula_df.loc[mask, 'loss_multiplier'] = np.clip(ula_df.loc[mask, 'loss_multiplier'], 0.8, 1.35 / (1 + loss_scale))
    if leave_out != 'Vehicle Age':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age
    if leave_out != 'npc':
        ula_df.loc[ula_df.kmx_npc_flag, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.high_pti_npc
    if leave_out != 'Student Loans':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.96 + 0.14 * ula_df.student_loan_flag
    if leave_out != 'high sales price':
        ula_df.loc[ula_df.mtn_model.isin([4.1]), 'loss_multiplier'] *= 0.98 + 0.22 * ula_df.high_sales_price_flag
    if leave_out != 'Driver flag':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.15 * ula_df.driver_flag
    if leave_out != 'Louisiana':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.35 * ula_df.louisiana_flag
    if leave_out != 'georgia':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.georgia_flag
    if leave_out != 'txca':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score > 140), 'loss_multiplier'] *= 1 - 0.1 * ula_df.txca_flag
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score <= 140) & (ula_df.cd_model_score >= 135), 'loss_multiplier'] *= 1 - 0.05 * ula_df.txca_flag
    if leave_out != 'state_counter_adj':
        mask = ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score < 150) & ~(ula_df.louisiana_flag | ula_df.georgia_flag | ula_df.txca_flag)
        ula_df.loc[mask, 'loss_multiplier'] *= 1.012
    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= (
            0.96 + (0.01 * ula_df.soft_pull_flag - 0.18 * ula_df.chime_flag * ula_df.soft_pull_flag) + 0.46 * ula_df.chime_flag)
    if leave_out != 'Job time':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag
    if leave_out != 'Existing DQ':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag
    if leave_out != 'Employment type':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag
    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag
    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.1 * (0.99 + 0.11 * ula_df.low_bureau_flag) * (0.978 + 0.172 * ula_df.open_tl_flag)
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= (1 * (0.98 + 0.22 * ula_df.low_bureau_flag) * (0.945 + 0.405 * ula_df.open_tl_flag) / np.maximum(ula_df.cd_perc_flag * ula_df.open_tl_flag * 1.2, 1))
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.97 + 0.15 * ula_df.cd_perc_flag
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.954 + 0.346 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.05 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.10 * ula_df.cd_perc_flag
    if leave_out != 'blanket adjustment':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] /= 1.1
    if leave_out != 'Clip':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] = np.clip(
            ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'], 0.75, 1.4)

    return ula_df

In [25]:
# =============================================================================
# CELL 6: DATA FETCH (SQL + PICKLE)
# =============================================================================
os.makedirs('cache', exist_ok=True)
force = run_every_query

need_conn = force or not all(
    os.path.exists(p) for p in ('../../cache/ula_v2.pkl', '../../cache/dla_v1.pkl', '../../cache/new_recovery_v1.pkl')
)

if need_conn:
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        ula_df_total = cached_sql(
            '../queries/vintage_level_ula_query_v2.txt', '../../cache/ula_v2.pkl',
            sub_list=[('{min_book_date}', min_date_sql)], connection=conn, force_refresh=force,
        )
        print('ULA ready')
        dla_df = cached_sql(
            '../queries/new_dll_query.txt', '../../cache/dla_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('DLA ready')
        new_recovery = cached_sql(
            '../queries/new_recovery_queryt.txt', '../../cache/new_recovery_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('New recovery ready')
else:
    ula_df_total = get_pickle('../../cache/ula_v2.pkl')
    dla_df = get_pickle('../../cache/dla_v1.pkl')
    new_recovery = get_pickle('../../cache/new_recovery_v1.pkl')
    print('ULA, DLA, New recovery loaded from cache')

print(f"ULA records: {len(ula_df_total):,}")

ULA, DLA, New recovery loaded from cache
ULA records: 5,020,205


In [26]:
# =============================================================================
# CELL 7: DATA PREP, FLAG CREATION, AND FILTERING
# =============================================================================

ula_df_total = ula_df_total[ula_df_total.lob != 'Core']

ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

for df in [ula_df_total, new_recovery]:
    df['quarter'] = pd.to_datetime(df[date_col]).dt.to_period('Q')
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')

for df in [ula_df_total, new_recovery]:
    df['book_week'] = df['book_week'].astype(str)
    df['app_week'] = df['app_week'].astype(str)

ula_df_total[f'{date_col}_str'] = ula_df_total[date_col].astype(str)
date_col_str = f'{date_col}_str'

# --- ULA Processing ---
ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

# --- DLA Merge ---
dla_df = dla_df.rename(columns={"valid_vintage": "book_vintage"})
dla_explicit = dla_df[dla_df['book_vintage'] != 'current']
dla_current = dla_df[dla_df['book_vintage'] == 'current'].drop(columns='book_vintage')
last_explicit_vintage = dla_explicit['book_vintage'].max()

ula_df_total = ula_df_total.merge(dla_explicit, how='left', on=['dealer_number', 'book_vintage'])
new_and_missing = ula_df_total['pricing_scalar'].isna() & (ula_df_total['book_vintage'] > last_explicit_vintage)
fallback = ula_df_total.loc[new_and_missing, ['dealer_number']].merge(dla_current, on='dealer_number', how='left')
for col in ['dll_edition', 'loss_ratio', 'dealer_level', 'pricing_scalar']:
    ula_df_total.loc[new_and_missing, col] = fallback[col].values

ula_df_total['pricing_scalar'] = ula_df_total['pricing_scalar'].fillna(1)
ula_df_total.loc[ula_df_total.frni_flag == 1, 'pricing_scalar'] *= 1.05
ula_df_total.loc[ula_df_total.frni_flag == 0, 'pricing_scalar'] *= 0.95

# --- Driver Flag ---
warnings.filterwarnings("ignore", category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('not provided')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings("default", category=UserWarning)

# --- ULA NA Handling ---
ula_df_total = ula_df_total.dropna(subset=['sale_price', 'cd_model_score', 'pti', 'lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# --- Filters ---
ula_df_total['bbltv'] = ula_df_total.amt_financed / ula_df_total.bbvalue.replace(0, np.nan)
ula_df_total = ula_df_total[ula_df_total.amt_financed <= 75000]
ula_df_total = ula_df_total[ula_df_total.pti <= 0.6]
ula_df_total = ula_df_total[
    (ula_df_total.lob == 'MCY') |
    (ula_df_total.bbltv <= 10.0) |
    (ula_df_total.bbvalue.isna()) |
    (ula_df_total.bbvalue == 0)
]
ula_df_total = ula_df_total[ula_df_total.total_income <= 200000]

print(f"ULA after filters: {len(ula_df_total):,}")

# --- NonKMX Flags ---
ula_df_total['ent_fld_flag'] = (ula_df_total.lob == 'ENT') | (ula_df_total.lob == 'FLD')
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500) & (ula_df_total.lob != 'MCY')
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY')
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.3) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col_str] >= '2022-07-01') & (ula_df_total[date_col_str] < '2025-01-01')
ula_df_total['mcy_low_mileage_flag'] = (ula_df_total.lob == 'MCY') & (ula_df_total.cd_model_score > 140) & (ula_df_total.mileage <= 20000) & (ula_df_total.vehicle_age <= 10)
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['pricing_change_flag'] = ula_df_total[date_col_str] >= '2024-10-01'
ula_df_total['ent_flag'] = (ula_df_total.lob == 'ENT')
ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col_str] >= '2023-05-01'
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & ula_df_total.lob.isin({'AN', 'FLD', 'FRN', 'STG'}) & (ula_df_total.cb_flag)
ula_df_total['student_loan_flag'] = np.where(ula_df_total.student_loan_flag == 1, 1, 0)

# --- KMX Flags ---
ula_df_total['high_sales_price_flag'] = ula_df_total.sale_price > 30000
ula_df_total['kmx_npc_flag'] = ula_df_total.kmx_npc_flag == 1
ula_df_total['high_pti_npc'] = ula_df_total.pti > 0.2
ula_df_total['job_time_flag'] = ula_df_total.employed_months < 6
ula_df_total['low_fico_flag'] = (ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 450)
ula_df_total['high_model_score_flag'] = ula_df_total.cd_model_score >= 146
ula_df_total['low_vantage_flag'] = (ula_df_total.fico_score < 300) & (ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450)
ula_df_total['louisiana_flag'] = ula_df_total.state == 'LA'
ula_df_total['txca_flag'] = ula_df_total.state.isin(['TX', 'CA'])
ula_df_total['georgia_flag'] = ula_df_total.state == 'GA'
ula_df_total['high_pti_tier_1_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.pti <= 0.25)
ula_df_total['high_pti_tier_2_flag'] = (ula_df_total.pti > 0.25) & (ula_df_total.pti <= 0.35)
ula_df_total['high_pti_tier_3_flag'] = ula_df_total.pti > 0.35
ula_df_total['existing_dq_flag'] = ula_df_total.existing_dq_count > 0
ula_df_total['kmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines >= 0.14
ula_df_total['low_bureau_flag'] = ((ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 475)) | ((ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450))
ula_df_total['soft_pull_flag'] = ula_df_total.pull_type.isin(['softpull', 'prequal'])
ula_df_total['cd_perc_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1)
ula_df_total['open_tl_flag'] = ula_df_total.open_tl == 0
ula_df_total['narrowed_soft_pull_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1) & ula_df_total.soft_pull_flag & (~ula_df_total.job_time_flag)

# --- Deduplicate driver flags ---
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

# --- MTN 4.1 model score transformation ---
is_mtn41 = ula_df_total.mtn_model == 4.1
ula_df_total.loc[is_mtn41, 'cd_model_score'] = (
    142 + (ula_df_total.loc[is_mtn41, 'cd_model_score'] - 142) * 1.5
)

# --- Period assignment and filtering ---
period_key = {'q': 'quarter', 'm': 'month'}
ula_df_total['period'] = ula_df_total[period_key[granularity]]
new_recovery['period'] = new_recovery[period_key[granularity]]

mask = (ula_df_total['period'] >= start_period) & (ula_df_total['period'] <= end_period)
ula_df_total = ula_df_total[mask]
nr_mask = (new_recovery['period'] >= start_period) & (new_recovery['period'] <= end_period)
new_recovery = new_recovery[nr_mask]

ula_df_total['vintage'] = format_vintage(ula_df_total['period'])
new_recovery['vintage'] = format_vintage(new_recovery['period'])

print(f"ULA refined: {len(ula_df_total):,} rows, {ula_df_total['vintage'].nunique()} vintages")
print(f"Recovery refined: {len(new_recovery):,} rows")

ULA after filters: 5,000,835
ULA refined: 879,897 rows, 27 vintages
Recovery refined: 858,176 rows


In [27]:
# =============================================================================
# CELL 8: ACCOUNT-LEVEL V3 RAGU SCORING (vectorized)
# =============================================================================

# --- Apply ULA multipliers (split by KMX vs nonKMX) ---
kmx_mask = ula_df_total.lob == 'KMX'
ula_kmx = get_ula_multiplier_kmx(ula_df_total[kmx_mask].copy())
ula_nonkmx = get_ula_multiplier_nonkmx(ula_df_total[~kmx_mask].copy())
ula_all = pd.concat([ula_kmx, ula_nonkmx])

# --- Merge recovery multiplier ---
nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(
    subset='account_number', keep='first')
acct_df = ula_all.merge(
    nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
    on='account_number', how='left'
).drop_duplicates(subset='account_number', keep='first')

has_bb = acct_df['bbvalue'].notna() & (acct_df['bbvalue'] > 0)
print(f"Total accounts: {len(acct_df):,}")
print(f"  with bbvalue > 0: {has_bb.sum():,}")
print(f"  with recovery: {acct_df['recovery_multiplier'].notna().sum():,}")

# --- Per-account V3 formula ---
acct_df['model_score'] = acct_df.cd_model_score
acct_df['ltv'] = np.where(has_bb, acct_df.amt_financed / acct_df.bbvalue, np.nan)
acct_df['con_term_filled'] = acct_df['con_term'].fillna(UL_TERM_CENTER)

acct_df['unit_loss'] = compute_unit_loss(acct_df['model_score'], acct_df['con_term_filled'])
acct_df['gross_loss_impact'] = compute_gross_loss_impact(acct_df['loss_multiplier'], acct_df['unit_loss'])
acct_df['recovery_impact'] = compute_recovery_impact(acct_df['recovery_multiplier'], acct_df['unit_loss'])

baseline_ltv_map = {lob: cfg['ltv'] for lob, cfg in BASELINES.items()}
baseline_apr_map = {lob: cfg['apr'] for lob, cfg in BASELINES.items()}
acct_df['baseline_ltv'] = acct_df.lob.map(baseline_ltv_map)
acct_df['baseline_apr'] = acct_df.lob.map(baseline_apr_map)

acct_df['ltv_impact'] = compute_ltv_impact(acct_df['baseline_ltv'], acct_df['ltv'])
acct_df['apr_impact'] = compute_apr_impact(acct_df['baseline_apr'], acct_df['apr'])

# --- RAGU score (pre-KMX rescaling) ---
acct_df['ragu_score'] = (
    acct_df['model_score']
    + acct_df['gross_loss_impact']
    + acct_df['recovery_impact']
    + acct_df['ltv_impact']
    + acct_df['apr_impact']
)

# --- KMX rescaling ---
kmx = acct_df.lob == 'KMX'
acct_df.loc[kmx, 'model_score'] = rescale_kmx(acct_df.loc[kmx, 'model_score'], is_model_score=True)
for col in ['gross_loss_impact', 'recovery_impact', 'ltv_impact', 'apr_impact']:
    acct_df.loc[kmx, col] *= KMX_BPS_PER_POINT
acct_df.loc[kmx, 'ragu_score'] = (
    acct_df.loc[kmx, 'model_score']
    + acct_df.loc[kmx, 'gross_loss_impact']
    + acct_df.loc[kmx, 'recovery_impact']
    + acct_df.loc[kmx, 'ltv_impact']
    + acct_df.loc[kmx, 'apr_impact']
)

# --- Select output columns ---
output_cols = [
    'account_number', 'lob', 'vintage', 'book_date', 'app_date', 'model_score',
    'gross_loss_impact', 'recovery_impact', 'ltv_impact', 'apr_impact',
    'ragu_score', 'amt_financed', 'ltv', 'apr', 'recovery_multiplier',
    'unit_loss', 'loss_multiplier', 'con_term_filled',
]
acct_df = acct_df[output_cols].copy()

print(f"\nAccount-level RAGU V3 computed: {len(acct_df):,} accounts")
print(f"  with full RAGU (bbvalue + recovery): {acct_df['ragu_score'].notna().sum():,}")
print(f"LOBs: {sorted(acct_df.lob.unique())}")
print(f"Vintages: {acct_df.vintage.nunique()}")
display(acct_df.head(10))

Total accounts: 874,256
  with bbvalue > 0: 787,165
  with recovery: 850,492

Account-level RAGU V3 computed: 874,256 accounts
  with full RAGU (bbvalue + recovery): 755,415
LOBs: ['', 'AN', 'ENT', 'FLD', 'FRN', 'KMX', 'MCY', 'ORL', 'STE', 'STG', 'unassigned']
Vintages: 27


,account_number,lob,vintage,book_date,app_date,model_score,gross_loss_impact,recovery_impact,ltv_impact,apr_impact,ragu_score,amt_financed,ltv,apr,recovery_multiplier,unit_loss,loss_multiplier,con_term_filled
0,9.012452e+10,KMX,2023 Q2,2023-05-30,2023-05-24,145.55,0.677154,10.337055,-0.007975,0.481046,157.037279,14730.62,1.601154,0.241,0.692948,0.306,0.908079,72.0
1,9.012456e+10,KMX,2023 Q3,2023-08-28,2023-08-26,167.00,2.050869,-16.689059,0.319702,-1.603485,151.078027,6942.90,1.142864,0.280,0.585196,-0.585,1.145624,72.0
2,9.012474e+10,KMX,2024 Q2,2024-06-03,2024-06-01,138.40,0.790277,17.103897,-0.438040,-1.603485,154.252650,36618.94,2.202643,0.280,0.581839,0.603,0.945561,72.0
3,9.012478e+10,KMX,2024 Q3,2024-07-31,2024-07-10,139.70,1.230591,12.946207,0.024526,-1.603485,152.297839,22596.52,1.555698,0.280,0.483722,0.549,0.906891,72.0
4,9.012449e+10,KMX,2023 Q1,2023-03-22,2023-03-20,136.45,0.716575,13.389678,0.052275,-1.603485,149.005043,19302.40,1.516888,0.280,0.401550,0.684,0.956483,72.0
5,9.012450e+10,KMX,2023 Q2,2023-04-13,2023-04-08,145.55,1.079116,8.835619,-0.597134,0.534495,155.402095,13944.63,2.425153,0.240,0.592299,0.306,0.853514,72.0
6,9.012458e+10,KMX,2023 Q3,2023-09-27,2023-09-26,144.90,2.567897,8.819755,-0.599617,0.481046,156.169080,12811.00,2.428626,0.241,0.543297,0.333,0.679680,72.0
7,9.012461e+10,KMX,2023 Q4,2023-11-20,2023-11-18,138.40,3.879844,18.625228,NaN,-1.603485,NaN,18777.08,NaN,0.280,0.633592,0.603,0.732732,72.0
8,9.012464e+10,KMX,2024 Q1,2024-01-20,2024-01-19,140.35,1.379265,18.035132,-0.308991,2.137980,161.593385,12840.69,2.022156,0.210,0.708719,0.522,0.890244,72.0
9,9.012456e+10,KMX,2023 Q3,2023-08-15,2023-07-27,139.70,1.876923,18.794055,-0.072840,3.206970,163.505109,22882.59,1.691874,0.190,0.702221,0.549,0.857988,72.0


In [28]:
# =============================================================================
# CELL 9: VINTAGE-LOB AGGREGATION + ROLLUPS + MMI ADJUSTMENT
# =============================================================================

def population_aware_agg(grp):
    """Aggregate metrics using correct non-NaN population for each metric."""
    result = {}

    full_pop = grp
    bb_pop = grp[grp['ltv'].notna()]
    scored_pop = grp[grp['recovery_impact'].notna() & grp['ltv'].notna()]

    for m in ['model_score', 'gross_loss_impact']:
        sub = full_pop[full_pop[m].notna()]
        if len(sub) > 0:
            result[m] = (sub[m] * sub['amt_financed']).sum() / sub['amt_financed'].sum()
        else:
            result[m] = np.nan

    for m in ['ltv_impact', 'apr_impact', 'ltv', 'apr']:
        sub = bb_pop[bb_pop[m].notna()]
        if len(sub) > 0:
            result[m] = (sub[m] * sub['amt_financed']).sum() / sub['amt_financed'].sum()
        else:
            result[m] = np.nan

    for m in ['recovery_impact', 'ragu_score']:
        sub = scored_pop[scored_pop[m].notna()]
        if len(sub) > 0:
            result[m] = (sub[m] * sub['amt_financed']).sum() / sub['amt_financed'].sum()
        else:
            result[m] = np.nan

    result['amt_financed'] = full_pop['amt_financed'].sum()
    return pd.Series(result)


vintage_lob_df = acct_df.groupby(['vintage', 'lob']).apply(
    population_aware_agg, include_groups=False
).reset_index()

print(f"Vintage-LOB aggregation: {len(vintage_lob_df)} rows "
      f"({vintage_lob_df.vintage.nunique()} vintages x {vintage_lob_df.lob.nunique()} LOBs)")

# --- Rollup aggregation ---
rollup_metrics = [
    'model_score', 'gross_loss_impact', 'recovery_impact',
    'ltv_impact', 'apr_impact', 'ragu_score', 'ltv', 'apr',
]
for group_name, group_lobs in ROLLUP_GROUPS.items():
    group = vintage_lob_df[vintage_lob_df.lob.isin(group_lobs)].copy()
    rollup = group.groupby('vintage').apply(
        weighted_average_and_sum, rollup_metrics, include_groups=False
    ).reset_index()
    rollup['lob'] = group_name
    vintage_lob_df = pd.concat([vintage_lob_df, rollup], ignore_index=True)

print(f"After rollups: {len(vintage_lob_df)} rows across {vintage_lob_df.lob.nunique()} groups")
print(f"Groups: {sorted(vintage_lob_df.lob.unique())}")

# --- MMI Market Adjustment (optional post-scoring overlay) ---
if MMI_ENABLED:
    mmi_raw = pd.read_csv(MMI_CSV, header=None, names=['month_str', 'mmi'])
    mmi_raw = mmi_raw.dropna(subset=['mmi'])
    mmi_raw['date'] = pd.to_datetime(mmi_raw['month_str'], format='%b-%y')
    mmi_raw = mmi_raw.sort_values('date').reset_index(drop=True)
    mmi_raw['mmi_at_repo'] = mmi_raw['mmi'].shift(-MMI_LAG_MONTHS)
    mmi_raw['mmi_pct_change'] = (mmi_raw['mmi_at_repo'] - mmi_raw['mmi']) / mmi_raw['mmi']

    if period_freq == 'Q':
        mmi_raw['vintage'] = (mmi_raw['date'].dt.year.astype(str) + ' Q'
                              + mmi_raw['date'].dt.quarter.astype(str))
        mmi_agg = mmi_raw.groupby('vintage')['mmi_pct_change'].mean().reset_index()
    else:
        mmi_raw['vintage'] = (mmi_raw['date'].dt.year.astype(str) + ' M'
                              + mmi_raw['date'].dt.month.astype(str).str.zfill(2))
        mmi_agg = mmi_raw[['vintage', 'mmi_pct_change']].dropna()

    vintage_lob_df = vintage_lob_df.merge(mmi_agg, on='vintage', how='left')
    mmi_mask = vintage_lob_df['mmi_pct_change'].notna()
    vintage_lob_df['ragu_score_raw'] = vintage_lob_df['ragu_score'].copy()
    vintage_lob_df.loc[mmi_mask, 'ragu_score'] = (
        vintage_lob_df.loc[mmi_mask, 'ragu_score']
        + MMI_COEF * vintage_lob_df.loc[mmi_mask, 'mmi_pct_change']
    )
    print(f"MMI adjustment applied: {mmi_mask.sum()} vintage-LOB rows adjusted (coef={MMI_COEF:.2f})")
else:
    print("MMI adjustment: DISABLED")

# --- Verification: RAGU = MS + GL + Recovery + LTV + APR ---
sample = vintage_lob_df[vintage_lob_df.lob == 'POS'].head(5)
print(f"\nVerification (POS, first 5 periods):")
print(f"  RAGU formula: MS + GL + Recovery + LTV + APR")
for _, row in sample.iterrows():
    check = row['model_score'] + row['gross_loss_impact'] + row['recovery_impact'] + row['ltv_impact'] + row['apr_impact']
    print(f"  {row['vintage']}: {row['model_score']:.2f} + {row['gross_loss_impact']:.2f} "
          f"+ {row['recovery_impact']:.2f} + {row['ltv_impact']:.2f} + {row['apr_impact']:.2f} "
          f"= {check:.2f} (stored: {row['ragu_score']:.2f})")

display(vintage_lob_df.sort_values(['lob', 'vintage']).head(20))

Vintage-LOB aggregation: 207 rows (27 vintages x 11 LOBs)
After rollups: 288 rows across 14 groups
Groups: ['', 'AN', 'ENT', 'FLD', 'FRN', 'Franchise Independent', 'KMX', 'MCY', 'ORL', 'POS', 'STE', 'STG', 'nonKMX', 'unassigned']
MMI adjustment: DISABLED

Verification (POS, first 5 periods):
  RAGU formula: MS + GL + Recovery + LTV + APR
  2020 Q1: 135.81 + 1.62 + 21.57 + -0.07 + 0.30 = 159.23 (stored: 159.29)
  2020 Q2: 137.49 + 1.44 + 19.90 + -0.08 + 0.09 = 158.84 (stored: 158.89)
  2020 Q3: 138.64 + 1.37 + 18.40 + 0.06 + -0.02 = 158.46 (stored: 158.49)
  2020 Q4: 137.67 + 1.24 + 19.84 + -0.03 + 0.06 = 158.79 (stored: 158.78)
  2021 Q1: 136.89 + 1.47 + 20.83 + 0.07 + 0.03 = 159.29 (stored: 159.29)


,vintage,lob,model_score,gross_loss_impact,ltv_impact,apr_impact,ltv,apr,recovery_impact,ragu_score,amt_financed
0,2020 Q1,,131.527448,-0.729925,NaN,NaN,1.361088,0.266971,NaN,NaN,111357.46
9,2020 Q2,,138.304012,0.783636,NaN,NaN,1.709419,0.265249,NaN,NaN,67289.38
1,2020 Q1,AN,131.789364,1.138353,0.044817,0.574782,1.899257,0.243010,30.271721,163.657201,32280007.35
10,2020 Q2,AN,133.695827,0.964227,0.002124,0.074826,1.938069,0.249090,27.687186,162.402112,27949535.42
18,2020 Q3,AN,135.295362,0.247837,0.312686,0.316401,1.655740,0.246152,26.442076,162.578517,26877270.67
25,2020 Q4,AN,135.780174,0.645503,0.299981,0.813460,1.667290,0.240108,26.311453,163.815763,22254614.52
32,2021 Q1,AN,134.604445,0.994470,0.274212,0.164136,1.690716,0.248004,27.640112,163.689636,32911077.92
39,2021 Q2,AN,135.141784,0.974134,0.465689,0.303817,1.516646,0.246305,26.977457,163.835508,39823865.94
46,2021 Q3,AN,135.348538,0.705941,0.384910,0.153665,1.590082,0.248131,26.488113,163.114674,26626969.87
53,2021 Q4,AN,134.395671,0.368739,0.406841,0.123383,1.570145,0.248500,27.152967,162.350957,23219355.09


In [30]:
# =============================================================================
# CELL 10: EXCEL EXPORT (AGGREGATED ONLY -- no individual loan rows)
# =============================================================================
# NOTE: RAGU = Model Score + GL Impact + Recovery Impact + LTV Impact + APR Impact
#       All components are additive. Higher RAGU = lower expected loss.

SHEET_MAP = {'q': 'V3 Data (Q)', 'm': 'V3 Data (M)'}

METRIC_ROWS = [
    ('Model Score',           'model_score'),
    ('Gross Loss Impact',     'gross_loss_impact'),
    ('Recovery Impact',       'recovery_impact'),
    ('LTV Impact',            'ltv_impact'),
    ('APR Impact',            'apr_impact'),
    ('RAGU Score',            'ragu_score'),
    ('Amount Financed',       'amt_financed'),
    ('Weighted LTV',          'ltv'),
    ('Weighted APR',          'apr'),
]

sheet_name = SHEET_MAP[granularity]
sorted_vintages = sorted(vintage_lob_df['vintage'].unique())
all_export_lobs = LOBS + list(ROLLUP_GROUPS.keys())

if os.path.exists(EXCEL_OUTPUT):
    wb = openpyxl.load_workbook(EXCEL_OUTPUT)
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws = wb.create_sheet(sheet_name, 0)
else:
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = sheet_name

current_row = 1
for lob in all_export_lobs:
    lob_data = vintage_lob_df[vintage_lob_df.lob == lob].set_index('vintage')

    ws.cell(row=current_row, column=1, value=lob)
    for col_idx, v in enumerate(sorted_vintages, start=2):
        ws.cell(row=current_row, column=col_idx, value=v)
    current_row += 1

    for label, col_key in METRIC_ROWS:
        ws.cell(row=current_row, column=1, value=label)
        for col_idx, v in enumerate(sorted_vintages, start=2):
            if v in lob_data.index:
                ws.cell(row=current_row, column=col_idx, value=lob_data.loc[v, col_key])
        current_row += 1

    current_row += 1

wb.save(EXCEL_OUTPUT)
print(f"Sheet '{sheet_name}': {len(all_export_lobs)} groups x {len(sorted_vintages)} periods")
print(f"Saved to {EXCEL_OUTPUT}")
print(f"\nFormula: RAGU = Model Score + GL Impact + Recovery Impact + LTV Impact + APR Impact")

Sheet 'V3 Data (Q)': 9 groups x 27 periods
Saved to ragu_v3_production.xlsx

Formula: RAGU = Model Score + GL Impact + Recovery Impact + LTV Impact + APR Impact
